# 02 - Train the three models

Exactly the rules of `docs/LLD.md` sections 5.2 to 5.5: deterministic 20 % test rows, three independent
XGBoost regressors with frozen parameters, honest evaluation on the test rows only.

In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

import common as c

c.OUT_DIR.mkdir(exist_ok=True)
(c.OUT_DIR / "figures").mkdir(exist_ok=True)
(c.OUT_DIR / "models").mkdir(exist_ok=True)
pd.set_option("display.precision", 4)
df = c.load_seed()
train, test = c.split(df)

## 1. Test rows (holdout)
`is_holdout = int(sha256(f"seed:{CustomerID}").hexdigest()[:8], 16) % 5 == 0`: the same rows are always test rows.

In [2]:
print(f"training rows: {len(train)} | test rows: {len(test)} ({len(test) / len(df):.1%})")
assert (len(train), len(test)) == (6398, 1602)
assert [c.is_holdout("seed", i) for i in range(8000, 8020)] == [c.is_holdout("seed", i) for i in range(8000, 8020)]
assert set(train["external_id"]).isdisjoint(test["external_id"])

training rows: 6398 | test rows: 1602 (20.0%)


## 2. Inputs
10 numeric columns and one 0/1 column per category value; optional columns only when they have 2 or more values.

In [3]:
spec = c.build_feature_spec(train)
x_train, x_test = c.encode(train, spec), c.encode(test, spec)
print(len(spec.columns), "model columns")
print("optional fields used:", spec.optional_categories or "none (constant columns)")
assert len(spec.columns) == 21 and not spec.optional_categories
list(spec.columns)

21 model columns
optional fields used: none (constant columns)


['ad_spend',
 'age',
 'income',
 'previous_purchases',
 'loyalty_points',
 'pages_per_visit',
 'time_on_site',
 'social_shares',
 'email_opens',
 'email_clicks',
 'campaign_channel=Social Media',
 'campaign_channel=Email',
 'campaign_channel=PPC',
 'campaign_channel=SEO',
 'campaign_channel=Referral',
 'campaign_type=Awareness',
 'campaign_type=Consideration',
 'campaign_type=Conversion',
 'campaign_type=Retention',
 'gender=Male',
 'gender=Female']

## 3. Training
Parameters (`ml/params.py`): `n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
min_child_weight=1, reg_lambda=1.0, tree_method="hist", random_state=42, n_jobs=2`.

In [4]:
print(c.XGB_PARAMS)
models = c.train_models(x_train, train)
predictions = c.predict(models, x_test)

{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'reg_lambda': 1.0, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': 2}


## 4. Honest evaluation on the test rows
`baseline_mae` is the error of always predicting the training mean. A useful model must beat it clearly.

In [5]:
metrics = c.evaluate(predictions, test, train)
table = pd.DataFrame({t: metrics[t] for t in c.TARGETS}).T
table["beats_baseline"] = table["mae"] < table["baseline_mae"]
display(table)
print("overall reliability:", metrics["overall_reliability"], "| mean R2:", round(metrics["mean_r2"], 4))
(c.OUT_DIR / "metrics_seed.json").write_text(json.dumps(metrics, indent=2))

,mae,rmse,r2,baseline_mae,reliability,beats_baseline
click_through_rate,0.0733,0.0852,-0.0342,0.0725,low,False
conversion_rate,0.0485,0.0562,-0.0373,0.048,low,False
website_visits,12.3977,14.4797,-0.0223,12.3192,low,False


overall reliability: low | mean R2: -0.0313


641

### Expected outcome
The seed is fabricated, so every R2 is close to 0 (slightly negative) and the reliability is **low** for the three figures,
matching the example of `DATA_MODEL.md` 4.7. An R2 above 0.10 on this data would mean a leak or a bug.

In [6]:
for target in c.TARGETS:
    assert metrics[target]["r2"] < 0.10, f"{target}: suspicious R2 {metrics[target]['r2']:.3f} (leak?)"
    assert metrics[target]["reliability"] == "low"
assert metrics["overall_reliability"] == "low"
print("checks passed: no leak, reliability low for the three targets")

checks passed: no leak, reliability low for the three targets


## 5. What the models rely on (global importance)
Mean absolute contribution on the test rows, grouped by original field, normalised to 1 (LLD 6.6).
On data without signal the importance is spread almost evenly: it explains nothing.

In [7]:
importance = c.global_importance(models, x_test, spec)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
for ax, target in zip(axes, c.TARGETS):
    series = importance[target].sort_values()
    ax.barh(series.index, series.to_numpy(), color="#5b7a99")
    ax.set_title(target)
fig.tight_layout()
fig.savefig(c.OUT_DIR / "figures" / "02_global_importance.png", dpi=110)
plt.show()
for target in c.TARGETS:
    print(target, "top share:", importance[target].round(3).head(3).to_dict())
    assert abs(importance[target].sum() - 1) < 1e-9

click_through_rate top share: {'pages_per_visit': 0.127, 'income': 0.108, 'ad_spend': 0.107}
conversion_rate top share: {'loyalty_points': 0.138, 'ad_spend': 0.123, 'pages_per_visit': 0.115}
website_visits top share: {'ad_spend': 0.126, 'social_shares': 0.117, 'loyalty_points': 0.11}


## 6. Artifacts round trip
Models are saved as JSON (`save_model`), like the phase 7 bundle (LLD 5.6), then reloaded to prove the files are complete.

In [8]:
for target, model in models.items():
    model.save_model(c.OUT_DIR / "models" / f"{target}.json")
reloaded = {}
for target in c.TARGETS:
    model = xgb.XGBRegressor()
    model.load_model(c.OUT_DIR / "models" / f"{target}.json")
    reloaded[target] = model
again = c.predict(reloaded, x_test)
for target in c.TARGETS:
    np.testing.assert_allclose(again[target], predictions[target], rtol=1e-5, atol=1e-6)
print("reloaded models give the same predictions")

reloaded models give the same predictions


## 7. One example campaign
Outputs are bounded: rates in [0, 1], visits >= 0 (LLD 6.1). Leads and cost per result follow LLD 6.2.

In [9]:
example = test.iloc[[0]]
point = c.predict(models, c.encode(example, spec))
ctr, conv, visits = (float(point[t][0]) for t in c.TARGETS)
leads = c.estimated_leads(visits, conv)
cost = c.estimated_cost_per_result(float(example["ad_spend"].iloc[0]), leads)
print(example[["campaign_channel", "campaign_type", "ad_spend"]].to_dict("records")[0])
print(f"predicted CTR {ctr:.2%} | conversion rate {conv:.2%} | visits {visits:.1f} | leads {leads:.2f} | cost per result {cost:.2f}")

{'campaign_channel': 'Social Media', 'campaign_type': 'Awareness', 'ad_spend': 6497.870068417766}
predicted CTR 14.89% | conversion rate 11.24% | visits 27.3 | leads 3.07 | cost per result 2118.55


## Conclusion
The training chain follows the LLD rules and runs in well under a second per model. On the fabricated seed the models
are not better than a constant prediction (low reliability, as designed). Notebook 03 proves the chain itself is sound
with controls, including a dataset that really has a signal.